# 🎯 YOLO11n'i kendi veri setinle eğit: Box vs Wheel
**Edge AI serisi, Bölüm 6.** Bu defter Google Colab'da (**T4 GPU**) çalışır. Çıktısı olan `best.pt`, Raspberry Pi 5'e götürülüp NCNN'ye çevrilir.

**Başlamadan:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`

| Adım | Ne yapıyoruz? |
|---|---|
| 1 | Ultralytics'i kur, GPU'yu doğrula |
| 2 | Veri setini Roboflow'dan indir |
| 3 | Veri setini incele |
| 4 | YOLO11n'i 50 epoch eğit (`imgsz=320`) |
| 5 | Eğitim grafikleri ve confusion matrix |
| 6 | Test kümesinde tahmin |
| 7 | Gerçek eşikle (conf=0.5) doğrulama |
| 8 | `best.pt`'yi indir |

## 1) Kurulum ve GPU kontrolü
`ultralytics.checks()` çıktısında **CUDA:0 (Tesla T4)** görmelisin. `CPU` yazıyorsa çalışma zamanını değiştir.

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2) Veri setini indir
Veri seti: Roboflow Universe → `marcelo-rovai-riila / box-versus-wheel-auto-dataset`, **sürüm 8**, format **YOLOv11**.

> 🔐 **API anahtarını asla GitHub'a yükleme, ekran görüntüsünde gösterme, defteri herkese açık paylaşma.** Daha güvenli yöntem: Colab'ın sol menüsündeki 🔑 **Secrets** bölümüne `ROBOFLOW_API_KEY` adıyla ekle ve aşağıdaki gibi oku.

In [ ]:
%cd /content
!pip install -q roboflow
from roboflow import Roboflow

try:
    from google.colab import userdata
    API_KEY = userdata.get("ROBOFLOW_API_KEY")   # Colab Secrets'tan oku (önerilen)
except Exception:
    API_KEY = "YOUR_API_KEY"                      # ya da buraya yaz, ama ASLA paylaşma

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("marcelo-rovai-riila").project("box-versus-wheel-auto-dataset")
dataset = project.version(8).download("yolov11")

## 3) Veri setini incele
Kaç resim, kaç etiket, kaç boş (background) resim var?

In [ ]:
import os, glob
DS = dataset.location
for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{DS}/{split}/images/*")
    lbls = glob.glob(f"{DS}/{split}/labels/*.txt")
    bos = sum(1 for l in lbls if os.path.getsize(l) == 0)
    print(f"{split:6s}: {len(imgs):4d} resim | {len(lbls):4d} etiket dosyası | {bos} boş (background)")

print("\nÖrnek etiket satırı  [sınıf  x_merkez  y_merkez  genişlik  yükseklik]  (0–1 normalize):")
print(open(sorted(glob.glob(f"{DS}/train/labels/*.txt"))[0]).readline())

## 4) Eğitim
- `data.yaml`'ı **mutlak yollarla** yeniden yazıyoruz (Roboflow'un yazdığı göreli yollar bazen kırılır).
- `epochs=50`, `imgsz=320` → **imgsz kuralı:** Export ve çıkarımda da 320 kullanılacak!
- T4'te yaklaşık 2 dakika sürer.

In [ ]:
from ultralytics import YOLO

yaml_text = f"""path: {DS}
train: train/images
val: valid/images
test: test/images
nc: 2
names: ['box', 'wheel']
"""
open(f"{DS}/data.yaml", "w").write(yaml_text)

model = YOLO("yolo11n.pt")          # COCO'da önceden eğitilmiş → transfer learning
results = model.train(data=f"{DS}/data.yaml", epochs=50, imgsz=320, plots=True)

## 5) Eğitim grafikleri ve confusion matrix
- `train/*_loss` ve `val/*_loss` **birlikte** düşüyorsa overfitting yok.
- Confusion matrix varsayılan olarak **conf=0.001** ile çizilir; background sütunundaki sayılar düşük güvenli çöp kutulardır (7. hücreye bak).

In [ ]:
from IPython.display import Image, display
display(Image("/content/runs/detect/train/results.png", width=900))
display(Image("/content/runs/detect/train/confusion_matrix.png", width=500))

## 6) Test kümesinde tahmin
Modelin **hiç görmediği** 10 resim. `imgsz=320`, `conf=0.5`.

In [ ]:
best = YOLO("/content/runs/detect/train/weights/best.pt")
res = best.predict(f"{DS}/test/images", imgsz=320, conf=0.5, save=True, verbose=False)
for r in res[:3]:
    display(Image(r.save_dir + "/" + r.path.split("/")[-1], width=320))

## 7) Gerçek çalışma eşiğiyle doğrulama (conf=0.5)
Aynı model, aynı resimler; sadece eşik değişiyor → **precision–recall takası**.
- conf=0.001: ~70 yanlış alarm, 0 kaçırma
- conf=0.5: 0 yanlış alarm, 1 kaçırma

⚠️ mAP'i raporlarken **düşük eşikle** (varsayılan) ölç; conf=0.5'te PR eğrisi kesildiği için mAP yapay olarak düşer.

In [ ]:
m = best.val(data=f"{DS}/data.yaml", imgsz=320, conf=0.5, plots=True)
display(Image(m.save_dir / "confusion_matrix.png", width=500))

## 8) `best.pt`'yi indir
Sonra Windows'ta:
```powershell
scp best.pt pi@raspi-5.local:Documents/YOLO/models/box_wheel_320_yolo11n.pt
```
Pi'de:
```bash
yolo export model=models/box_wheel_320_yolo11n.pt format=ncnn imgsz=320
```

In [ ]:
from google.colab import files
files.download("/content/runs/detect/train/weights/best.pt")

# (İsteğe bağlı) Test resimlerini Pi'ye götürmek için zip'le
!zip -j -q /content/box_wheel_test.zip {DS}/test/images/*
files.download("/content/box_wheel_test.zip")